<a href="https://colab.research.google.com/github/Heng1222/Ohsumed_classification/blob/Lora_with_2002MeSH/Model/Lora_v3(replaceMeSH2002).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from tqdm import tqdm

def download_mesh(url, filename):
    # 使用 stream=True 啟動串流模式
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(filename, 'wb') as file, tqdm(
        desc=filename,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024 * 1024): # 每次讀取 1MB
            size = file.write(data)
            bar.update(size)

url = "https://nlmpubs.nlm.nih.gov/projects/mesh/1999-2010/xmlmesh/desc2002.xml"
download_mesh(url, "desc2002.xml")
print("下載完成！")

desc2002.xml: 100%|██████████| 210M/210M [00:04<00:00, 48.4MiB/s]


下載完成！


In [ ]:
# !wget -c https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2026.xml

--2026-03-25 06:12:49--  https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2026.xml
Resolving nlmpubs.nlm.nih.gov (nlmpubs.nlm.nih.gov)... 130.14.173.134
Connecting to nlmpubs.nlm.nih.gov (nlmpubs.nlm.nih.gov)|130.14.173.134|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 312952703 (298M) [text/xml]
Saving to: ‘desc2026.xml’

desc2026.xml        100%[===================>] 298.45M  19.4MB/s    in 11s     

2026-03-25 06:13:00 (27.8 MB/s) - ‘desc2026.xml’ saved [312952703/312952703]



In [ ]:
def parse_mesh_hierarchy(xml_file):
    mesh_dict = {}
    # 增加 recover=True 讓 parser 嘗試忽略微小的 XML 語法錯誤
    parser = etree.XMLParser(recover=True)
    try:
        context = etree.iterparse(xml_file, events=('end',), tag='DescriptorRecord', parser=parser)
        for _, elem in context:
            try:
                name_node = elem.find('.//DescriptorName/String')
                tree_numbers = [tn.text for tn in elem.findall('.//TreeNumber')]
                if name_node is not None and tree_numbers:
                    mesh_dict[name_node.text.lower()] = tree_numbers
            finally:
                # 確保無論是否成功處理該節點，都會釋放記憶體
                elem.clear()
                while elem.getprevious() is not None:
                    del elem.getparent()[0]
    except etree.XMLSyntaxError as e:
        print(f"解析中斷，目前已讀取 {len(mesh_dict)} 個標目。錯誤：{e}")
    return mesh_dict

In [ ]:
import lxml.etree as etree
import math
import pandas as pd
from itertools import combinations

def parse_mesh_hierarchy(xml_file):
    # 建立 標目 -> 樹狀號 的映射
    mesh_dict = {}
    context = etree.iterparse(xml_file, events=('end',), tag='DescriptorRecord')

    for _, elem in context:
        name_node = elem.find('.//DescriptorName/String')
        tree_numbers = [tn.text for tn in elem.findall('.//TreeNumber')]
        if name_node is not None and tree_numbers:
            mesh_dict[name_node.text.lower()] = tree_numbers
        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]
    return mesh_dict

def get_wup_similarity(name1, name2, mesh_dict):
    # 取兩個標目所有路徑組合中相似度的最小值 (Min-mapping)
    t1_list = mesh_dict.get(name1, [])
    t2_list = mesh_dict.get(name2, [])
    if not t1_list or not t2_list: return 0.0

    # 關鍵：初始值必須設為可能的最大值 (1.0)
    min_sim = 1.0

    for t1 in t1_list:
        p1 = t1.split('.')
        for t2 in t2_list:
            p2 = t2.split('.')
            # 計算最深共同祖先 (LCS)
            lcs_depth = 0
            for i in range(min(len(p1), len(p2))):
                if p1[i] == p2[i]: lcs_depth += 1
                else: break

            # WUP 公式
            sim = (2.0 * lcs_depth) / (len(p1) + len(p2))

            # 改為取最小值
            min_sim = min(min_sim, sim)

    return min_sim

# 執行解析並生成訓練對 (以 Ohsumed 常見疾病類 C 類為範例)
mesh_data = parse_mesh_hierarchy('desc2002.xml')
target_words = [w for w, t in mesh_data.items() if any(tn.startswith('C') for tn in t)][:100] # 取前100個疾病詞做範例
word_pairs = []
for w1, w2 in combinations(target_words, 2):
    sim = get_wup_similarity(w1, w2, mesh_data)
    if sim > 0: # 僅保留有階層關係的詞對
        word_pairs.append({'word1': w1, 'word2': w2, 'wup_sim': sim})

train_df = pd.DataFrame(word_pairs)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizer
from peft import LoraConfig, get_peft_model
from lxml import etree
import pandas as pd
from itertools import combinations
import os
from transformers import Trainer, TrainingArguments

# ==========================================
# 1. 資料準備階段 (MeSH 解析)
# ==========================================

def parse_mesh_hierarchy(xml_file):
    mesh_dict = {}
    if not os.path.exists(xml_file):
        raise FileNotFoundError(f"找不到檔案: {xml_file}")

    context = etree.iterparse(xml_file, events=('end',), tag='DescriptorRecord')
    for _, elem in context:
        name_node = elem.find('.//DescriptorName/String')
        tree_numbers = [tn.text for tn in elem.findall('.//TreeNumber')]
        if name_node is not None and tree_numbers:
            mesh_dict[name_node.text.lower()] = tree_numbers
        elem.clear()
        while elem.getprevious() is not None:
            del elem.getparent()[0]
    return mesh_dict

def get_wup_similarity(name1, name2, mesh_dict):
    t1_list = mesh_dict.get(name1, [])
    t2_list = mesh_dict.get(name2, [])
    if not t1_list or not t2_list: return 0.0

    min_sim = 1.0 # 依照您的需求取所有路徑組合的最小值
    for t1 in t1_list:
        p1 = t1.split('.')
        for t2 in t2_list:
            p2 = t2.split('.')
            lcs_depth = 0
            for i in range(min(len(p1), len(p2))):
                if p1[i] == p2[i]: lcs_depth += 1
                else: break
            sim = (2.0 * lcs_depth) / (len(p1) + len(p2))
            min_sim = min(min_sim, sim)
    return min_sim

# 執行解析
print("正在解析 MeSH XML...")
mesh_data = parse_mesh_hierarchy('desc2002.xml')
# 篩選 C 類 (疾病) 前 100 個詞進行示範
target_words = [w for w, t in mesh_data.items() if any(tn.startswith('C') for tn in t)][:100]

word_pairs = []
for w1, w2 in combinations(target_words, 2):
    sim = get_wup_similarity(w1, w2, mesh_data)
    if sim > 0:
        word_pairs.append({'word1': w1, 'word2': w2, 'wup_sim': sim})

train_df = pd.DataFrame(word_pairs)
print(f"生成的訓練對數量: {len(train_df)}")

# ==========================================
# 2. 定義 Dataset 與 模型架構
# ==========================================

class SemanticInjectionDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # 這裡回傳字串，之後在訓練迴圈中再進行 Tokenization
        return row['word1'], row['word2'], torch.tensor(row['wup_sim'], dtype=torch.float)

class WordNetSemanticLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, v1, v2, target_wup):
        cos_sim = torch.nn.functional.cosine_similarity(v1, v2)
        return self.mse(cos_sim, target_wup)

# ==========================================
# 3. 初始化訓練環境
# ==========================================

model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
base_model = RobertaModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["query", "key", "value", "output.dense"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(base_model, lora_config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 將生成的 train_df 放入 Dataset
dataset = SemanticInjectionDataset(train_df, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = WordNetSemanticLoss()

# ==========================================
# 4. 訓練迴圈
# ==========================================

num_epochs = 50
patience = 5
best_loss = float('inf')
counter = 0

print(f"開始在 {device} 上進行訓練...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for w1, w2, target_sim in dataloader:
        target_sim = target_sim.to(device)
        optimizer.zero_grad()

        # Tokenization 處理
        inputs1 = tokenizer(list(w1), return_tensors="pt", padding=True, truncation=True).to(device)
        inputs2 = tokenizer(list(w2), return_tensors="pt", padding=True, truncation=True).to(device)

        # 取得 Mean Pooling 向量
        v1 = model(**inputs1).last_hidden_state.mean(dim=1)
        v2 = model(**inputs2).last_hidden_state.mean(dim=1)

        loss = criterion(v1, v2, target_sim)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # 早停與儲存
    if avg_loss < best_loss:
        best_loss = avg_loss
        counter = 0
        model.save_pretrained("./best_model_weights")
    else:
        counter += 1
        if counter >= patience:
            print(f"觸發早停！")
            break

    # 定期上傳
    if (epoch + 1) % 5 == 0:
        print("正在同步至 Hugging Face...")
        try:
            model.push_to_hub(repo_name, private=True)
            tokenizer.push_to_hub(repo_name)
        except Exception as e:
            print(f"上傳失敗: {e}")

print("訓練結束。")

正在解析 MeSH XML...
生成的訓練對數量: 116


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


開始在 cuda 上進行訓練...
Epoch 1/50, Loss: 0.2370
Epoch 2/50, Loss: 0.0723
Epoch 3/50, Loss: 0.0261
Epoch 4/50, Loss: 0.0199
Epoch 5/50, Loss: 0.0199
正在同步至 Hugging Face...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  26%|##5       | 3.94MB / 15.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Epoch 41/50, Loss: 0.0030
Epoch 42/50, Loss: 0.0033
Epoch 43/50, Loss: 0.0036
Epoch 44/50, Loss: 0.0033
Epoch 45/50, Loss: 0.0030
觸發早停！
訓練結束。
